# 01 · AutoErrorAnalyzer — build the pool

*Error type in a learner sentence (4 classes), or error detection (2 classes)*

### Where this sits

```
▶ 01 build the pool  →  02 sample  →  03 annotate  →  04 prompt  →  05 report
```

You run **01 once per group**, for your own track only. It ends by writing `data/pools/<track>_pool.json` — the file notebook 02 opens.

---

**What it is.** ~100 Japanese-EFL essays, annotated with a 26-category error taxonomy (Krippendorff's α ≈ .92). The file also holds **the published tool's own predictions**, so this is the one track where you can benchmark your LLM against both a human gold standard *and* an existing system.

**Difficulty of the labeling judgment:** ★★★ — hard. Many error types, and a sentence can carry several at once.

**Licence:** CC BY 4.0  
**Cite:** Mizumoto, A. (2025). *Studies in Second Language Acquisition, 47*(3), 867–884. OSF: osf.io/jyf3r

---

Every dataset in this course is reshaped into the **same canonical schema**, so one pipeline works for all of them:

```json
[{"id": 1, "text": "...", "label": "..."}]
```

The *raw* data, though, looks different every time. **That difference is the lesson** — half of building a gold standard is getting messy real data into a clean, consistent shape.

Those three keys are required on every track. Two tracks add more: `cars50` and `raamove` ask what a sentence *does in a passage*, which is not always decidable from the sentence on its own, so their items also carry `doc_id`, `sent_index`, `n_sents` and `context`. Extra keys are safe everywhere — nothing in the pipeline checks for keys it does not need.

> The reshaping code below is read straight out of `scripts/reshape.py` — it is the same code `scripts/prep_datasets.py` runs, not a copy of it. What is *missing* from it is missing on purpose: the ✏️ cells are the decisions, and they are yours. (Generated by `scripts/_generate_pool_notebooks.py`; edit that or `reshape.py`, never the `.ipynb`.)

## Setup — run this first

This cell mounts your Google Drive and finds your group's shared folder, `lda2-final-template`. Everything the project produces — the pool, the gold set, your prompts, the outputs — is an ordinary file in there, which is what makes it survive the runtime resetting *and* lets the rest of your group see it.

**One member sets the folder up once:**

1. That member runs the `git clone` line this cell prints if the folder is missing, which puts it in their own Drive.
2. They share it with the group (right-click ▸ *Share*), with edit access.
3. Everyone else opens *Shared with me*, right-clicks the folder, and chooses **Add shortcut to Drive** ▸ *My Drive*.

Keep that shortcut's name exactly `lda2-final-template`. It is what makes the same path work for all of you — if Drive renames it to `lda2-final-template (1)`, this cell will not find it.

From then on, open notebooks from the folder itself (*File ▸ Open notebook ▸ Drive*) rather than from the GitHub badge, so you are working on your group's copy and not a fresh one.

In [ ]:
# ------------------------------------------------------------------
# SETUP — run me first. You are not expected to read it.
# ------------------------------------------------------------------
# This cell is plumbing, and it is the only cell in the project that is.
# It finds your group's shared folder in Google Drive, because everything
# this project keeps goes in there: a Colab runtime is wiped when it resets,
# and nobody else in your group can see inside it. Then it makes the
# project's own code importable. Run it and move on; nothing below asks you
# to have understood it.

FOLDER = "lda2-final-template"     # the shared folder, in every member's Drive

import os, sys

PROJECT = ".."                              # running locally: it is just above us

try:
    from google.colab import drive           # only exists inside Colab
except ImportError:
    pass
else:
    drive.mount("/content/drive")
    PROJECT = "/content/drive/MyDrive/" + FOLDER
    if not os.path.isdir(PROJECT):
        raise RuntimeError(
            "Could not find " + PROJECT + "\n\n"
            "Setting the folder up for your group? Run this in a new cell:\n"
            "  !git clone https://github.com/egumasa/lda2-final-template.git "
            + PROJECT + "\n"
            "then share the folder with the rest of your group.\n\n"
            "Someone else already did? Open Drive, find the folder under "
            "'Shared with me', right-click it, and choose 'Add shortcut to "
            "Drive'. Keep the name exactly " + FOLDER + ".")
    # Work in the RUNTIME, not in Drive: the next cells download a whole
    # corpus, and raw data is big, mostly not ours to redistribute, and one
    # command to fetch again. The pool you build from it is what persists.
    os.makedirs("/content/raw", exist_ok=True)
    os.chdir("/content/raw")

# scripts/ and config.py, by their real paths - so they are found from wherever
# this notebook happens to be working.
sys.path.append(PROJECT)
sys.path.append(PROJECT + "/scripts")

# Re-read config.yaml every time this cell runs. Without the reload, Python
# hands back the settings it read the FIRST time, and editing config.yaml
# would appear to do nothing until you restarted the runtime.
import importlib
import config
importlib.reload(config)

# Named one by one rather than with `import *`, so that every name a cell
# below uses can be traced back to the file it came from — config.yaml for
# these, scripts/ for the rest.
from config import (TRACK, GROUP, RUN, SEED, N_PER_CLASS, DEV, CODERS,
                    MEMBERS, LABELS_ORDER, ROOT, OUT_DIR,
                    POOL_PATH, DEMO_POOL_PATH, SAMPLE_PATH, GOLD_PATH,
                    SAMPLE_BEFORE_TOPUP_PATH,
                    DEV_PATH, TEST_PATH, DISAGREED_PATH, PRED_PATH,
                    ROUNDS_PATH, TESTLOG_PATH,
                    PROMPT_FILE, SHEET_PATH, TRIAGE_PATH, describe)
from pathlib import Path

describe()                  # what this notebook is working on


## Step 1 — Download the raw data

The annotations live on the paper's **OSF** project. Now we fetch one CSV directly by its OSF link.

In [ ]:
import urllib.request

RAW_FILE = "data_category.csv"
urllib.request.urlretrieve("https://osf.io/download/gezat/", RAW_FILE)
print("downloaded", RAW_FILE)

## Step 2 — Look at the raw format

A **CSV**. The columns that matter are `Sentence`, `Human_ErrorCategories` (the gold) and `AEA_ErrorCategories` (the tool's prediction). A sentence can carry several comma-separated error codes, or `NO_ERROR`.

The cell below prints **every code in the file, with its frequency**. You need that list in front of you for step 3 — it is the taxonomy you are about to collapse.

In [ ]:
import csv

# DictReader hands you each row as {column name: value}.
with open(RAW_FILE, encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)
    print("columns:", reader.fieldnames)

Now we count how often each error code appears, which is the taxonomy you are about to collapse in step 3.

One row can carry several codes, comma-separated. A row with no error has nothing in that column at all, which Python reads as `None` — so `or ""` below stands in an empty string for it, because you cannot split `None`.

In [ ]:
codes = {}
with open(RAW_FILE, encoding="utf-8-sig", newline="") as f:
    for row in csv.DictReader(f):
        human_field = row["Human_ErrorCategories"] or ""
        for code in human_field.split(","):
            code = code.strip()
            if code:
                if code not in codes:
                    codes[code] = 0
                codes[code] = codes[code] + 1

print(len(codes), "distinct codes:")
for code, n in sorted(codes.items(), key=lambda kv: -kv[1]):
    print("   ", code, n)

### Reading CSV with `csv.DictReader`

`csv.DictReader` reads a CSV using its header row, so each row arrives as a dict keyed by column name — `row["Sentence"]` rather than `row[3]`. That is what the cell above used to print `fieldnames` and count the codes.

**The columns that matter:**

* `Sentence` — the text
* `Human_ErrorCategories` — the human annotation, and your gold. One sentence can carry **several comma-separated codes**, or the single marker `NO_ERROR`.
* `AEA_ErrorCategories` — the published tool's own prediction, which is what lets this track compare an LLM against an existing system as well as against humans

The reshaping function below uses exactly these:

1. `open(..., encoding="utf-8-sig")` — this file ships with a byte-order mark. Without `-sig` it gets glued to the first column name and every lookup on it fails.
2. `csv.DictReader(handle)` — iterate rows as `{column: value}` dicts.
3. `(row.get(col) or "").strip()` — `.get` survives a missing column and the `or ""` survives an empty cell, which would otherwise be `None`.
4. `human_field.split(",")` — one sentence's codes into a list, which `_l2_coarse_label` then maps through **your** `L2_COARSE` grouping.

## Step 3 — Reshape into the canonical schema

Two decisions, and the first is the biggest single judgment call in any of the four tracks:

1. ✏️ **Collapse the ~23 codes into a handful of categories.** The full taxonomy is too fine-grained to prompt for reliably at this scale, so you group it. Where you draw those boundaries decides what your study is *about*: Grammatical / Lexical / Mechanical / No error is one defensible cut, and it is the one this track was written around — but it is not the only one. Is a wrong preposition grammatical or lexical? Is spelling mechanical, or a lexical problem wearing a mechanical hat? Whatever you decide, **every code you do not list is a sentence you throw away**.
2. **Drop mixed-category sentences** — a sentence whose codes span more than one of your categories gets no label, so the task stays single-label. That is in the code below. It also means the dataset under-represents exactly the messiest sentences, which **belongs in your limitations section**.

You also get a binary detection version for free (any error at all: yes/no), which does not depend on your grouping at all.

> Read `_l2_coarse_label` below before you write your mapping. It returns `None` — i.e. drops the sentence — when the codes span more than one of your categories, so a *coarser* grouping keeps more data and a *finer* one keeps less. That trade-off is yours to make and to report.

## Step 3a — Group the error codes

Now we collapse the published taxonomy into the handful of categories your group will annotate: one line per code from step 2 that you want to keep, with your category name on the right. `NO_ERROR` is handled separately, so do not list it. This is the decision on this track.

**A code you leave out is not an error — it is a dropped sentence.** Compare your total in step 4 against the detection count to see how many.

The four category names in the cell are the cut this track was written around, and they are only a starting point. Rename them, merge them, or use two categories instead of four — what matters is that you can say why.

Put the grouping in `PLAN.md` as a table, with a one-line justification for any code a reasonable person would file somewhere else. That table is report section 1.

In [ ]:
# ✏️ Step 3a · Group the error codes ─────────────────────────────
# Maps each raw error code you are keeping to the broader category you will
# study, and prints how many codes and categories that leaves you with.
# Creates: L2_COARSE (a dict)

# ✏️ this runs as written — the work is deciding whether it should

# One line per code you are KEEPING. Scroll up to step 2 for the full list
# with frequencies — the frequent codes are the ones worth arguing over.
#
# The right-hand side is your category name, and repeating one is how you
# merge: every code you send to "Grammatical" becomes one class.

L2_COARSE = {
    "ART": "Grammatical",      # articles — given as an example of the shape
    "SP": "Mechanical",        # spelling — mechanical, or lexical? your call
    "PREP": "Grammatical",     # prepositions — grammatical, or lexical?
    "TENSE": "Grammatical",    # verb tense
    "N": "Lexical",            # wrong noun choice
    # … keep going, one line per code from step 2 that you want to study.
    # Four lines is not a study. Argue about each one before you add it.
}

print(len(L2_COARSE), "codes kept ·", sorted(set(L2_COARSE.values())))


### The code that does it — read it, then run it

Four functions. The first two read the `L2_COARSE` you just defined, so if you skip the cell above they will stop on `NameError: L2_COARSE is not defined`.

It is read straight out of `scripts/reshape.py` when this notebook is generated, so it is not a simplified copy: it is the code that runs.

It arrives one function per cell, so you can take them one at a time. **None of these cells print anything.** They only give the functions their names — that is what `def` does. You will see no output until the cell *after* them, which calls one.

First, the two library modules the code below needs. `import` is how Python is told to load one.

In [ ]:
import csv

`_l2_coarse_label` turns a sentence's comma-separated codes into ONE broader category, or `None` when the codes span more than one — those get dropped. The leading underscore is a convention meaning "a helper for the function below"; it is not a typo, and nothing stops you calling it.

In [ ]:
def _l2_coarse_label(human_field: str) -> str | None:
    """Collapse a sentence's comma-separated error codes to ONE broader category.

    Args:
        human_field: the comma-separated code list, e.g. "ART,SP".

    Returns:
        The broader category, or None when the sentence cannot get a single clean
        label - either it has no codes, or its codes span more than one category.
        Those get dropped, which keeps this a single-label task. It also means the
        dataset under-represents exactly the messiest sentences, which is worth a
        line in your limitations section.
    """
    ### Split the comma-separated code list ###
    codes = []
    for code in human_field.split(","):          # "ART,SP" -> ["ART", "SP"].
        code = code.strip()
        if code:                                 # Drop empties from a trailing comma.
            codes.append(code)

    ### Two easy cases first ###
    if not codes:                                # Nothing to go on -> no label.
        return None
    if codes[0] == "NO_ERROR":                   # The explicit "this sentence is clean" marker.
        return "No error"

    ### Map every code to its broader category ###
    categories = set()
    for code in codes:
        if code in L2_COARSE:                    # Codes you did not list are silently ignored here...
            categories.add(L2_COARSE[code])

    ### One category, or nothing ###
    if len(categories) == 1:                     # All the errors agree -> that is the label.
        return categories.pop()
    return None                       # no codes we recognise, or a mixed-category sentence

`reid` renumbers items 1, 2, 3 … so that every item has an id of its own.

In [ ]:
def reid(items: list[dict[str, str]]) -> list[dict[str, str]]:
    """Renumber ids sequentially from 1, keeping the current order.

    Args:
        items: the items to renumber. They are copied, not changed in place.

    Returns:
        The same items with new ids.
    """
    renumbered = []
    next_id = 1

    for item in items:
        ### Copy before writing ###
        new_item = dict(item)                    # Work on a copy, so the caller's item is left alone.

        ### Stamp the id ###
        new_item["id"] = next_id                 # Overwrite whatever id was there with the running number.
        renumbered.append(new_item)              # Keep it in the order it arrived.
        next_id = next_id + 1                    # Advance, so the next item gets a fresh id.

    return renumbered

`reshape_l2_errors` is the work: one pass over the CSV, two lists out — your categories, and the yes/no detection version.

In [ ]:
def reshape_l2_errors(csv_path: str | Path) -> tuple:
    """Read data_category.csv into TWO datasets: 4-way categories, and yes/no detection.

    The CSV also carries `AEA_ErrorCategories` - the published tool's own predictions -
    so this is the one track where you can benchmark your LLM against both a human gold
    standard AND an existing system.

    Args:
        csv_path: the downloaded data_category.csv.

    Returns:
        Two lists: the 4-way category items and the yes/no detection items.
    """
    category_rows = []
    detection_rows = []
    # utf-8-sig: the file ships with a byte-order mark, which would otherwise end up
    # glued to the first column name and break the lookup.
    with open(csv_path, encoding="utf-8-sig", newline="") as handle:

        ### Read the CSV a row at a time ###
        for record in csv.DictReader(handle):    # DictReader gives each row as {column: value}.

            ### Pull the two columns that matter ###
            sentence = (record.get("Sentence") or "").strip()                  # The text.
            human = (record.get("Human_ErrorCategories") or "").strip()        # The human codes, comma-separated.
            if not sentence or not human:        # No text or no annotation -> unusable either way.
                continue

            ### Dataset 1: the coarse category ###
            label = _l2_coarse_label(human)      # Returns None for mixed-category sentences...
            if label is not None:                # ...and those get no row here at all.
                category_rows.append({"id": 0, "text": sentence, "label": label})

            ### Dataset 2: did it have ANY error? ###
            if human == "NO_ERROR":              # This one does not depend on your grouping,
                detection_label = "No error"     # so every annotated sentence gets a row.
            else:
                detection_label = "Has error"
            detection_rows.append({"id": 0, "text": sentence, "label": detection_label})

    return reid(category_rows), reid(detection_rows)   # Two datasets, each with ids running 1..N.

`validate` checks that every item has an id, a text and a label. Nothing calls it here — step 5 does, just before saving.

In [ ]:
def validate(items: list[dict[str, str]],
             allowed: list[str] | None = None) -> None:
    """Check the canonical schema, and raise on the first problem found.

    Deliberately explicit rather than `assert`: assertions vanish under `python -O`,
    and a silently unvalidated dataset is exactly the kind of thing that surfaces as a
    baffling metric three days later.

    Args:
        items: the items to check, each needing "id", "text" and "label".
        allowed: the labels the scheme allows. Left out, any label passes.

    Returns:
        Nothing.

    Raises:
        ValueError: on the first item that is missing a field, has a repeated id, or
            carries a label outside `allowed`.
    """
    seen_ids = set()
    for position, item in enumerate(items):
        where = "Item number " + str(position + 1) + " of " + str(len(items))
        for field in ("id", "text", "label"):
            if field not in item:
                raise ValueError(
                    where + " has no `" + field + "`, and every item needs all three of "
                    "id, text and label.\n"
                    "That item was built by the reshaping step above, so go back to the "
                    "cell where you filled in your own decision and check it puts a "
                    "`" + field + "` on every row.")
        if item["id"] in seen_ids:
            raise ValueError(
                "Two items have the same id (" + str(item["id"]) + "), so one would "
                "overwrite the other in your annotation sheet.\n"
                "reid() renumbers everything 1, 2, 3 - make sure the last line of your "
                "reshaping step hands its rows to it.")
        seen_ids.add(item["id"])
        if not isinstance(item["text"], str) or not item["text"].strip():
            raise ValueError(
                "The item with id " + str(item["id"]) + " has no text - there is nothing "
                "there for a coder or the model to read.\n"
                "Blank rows usually come from the raw file. Skip them in the reshaping "
                "step rather than annotate them.")
        if not isinstance(item["label"], str) or not item["label"]:
            raise ValueError(
                "The item with id " + str(item["id"]) + " has no label.\n"
                "Every item in a pool needs the published label it came with. If your "
                "label mapping does not cover some code in the raw data, either add it "
                "or drop those rows in the reshaping step - do not leave the label blank.")
        if allowed is not None and item["label"] not in allowed:
            raise ValueError(
                "The item with id " + str(item["id"]) + " has the label '"
                + str(item["label"]) + "', which is not one of the labels you allowed:\n"
                "  " + ", ".join(sorted(allowed)) + "\n"
                "Either add it to your label set, or map it onto one of these in the "
                "cell where you wrote your label mapping.")

## Step 3b — Run it

Now we run the reshaping over the CSV. It hands back **two** lists at once — that is what the comma on the left of the `=` means.

In [ ]:
category_rows, detection_rows = reshape_l2_errors(RAW_FILE)
print("categories:", len(category_rows), " detection:", len(detection_rows))

## Step 3c — Choose your task

Now we pick which of the two tasks your group will study — the n-way categorisation from 3a, or the yes/no detection task — and give it the name `rows` that the rest of the notebook uses.

Detection is a genuinely easier task and a smaller project. If you take it, plan an extension: benchmarking against the published tool's own predictions is right there in the CSV.

In [ ]:
# ✏️ Step 3c · Choose your task ──────────────────────────────────
# Names one of the two tasks as the one you will study, and prints how many
# items it has.
# Creates: rows (a list) — either category_rows or detection_rows

# ✏️ this runs as written — the work is deciding whether it should

# Change this one word to "detection" for the yes/no version.
# It is written as a choice rather than two lines you delete one of,
# because with two live lines the SECOND one silently wins — and you
# would not find out until notebook 03, halfway through annotating.
TASK = "category"

if TASK == "category":
    rows = category_rows      # your categories from 3a
elif TASK == "detection":
    rows = detection_rows     # Has error / No error
else:
    raise ValueError(
        "TASK has to be either \"category\" or \"detection\", and it says "
        + repr(TASK) + ". Fix the line above and run this cell again.")

print("studying the", TASK, "task:", len(rows), "items")


## Step 4 — Check the label balance

Note how many sentences were dropped: compare your category count against the detection count, which keeps everything. The gap is your mixed-category sentences, and it is a direct consequence of the grouping you wrote in 3a.

Now we count what we have got: how many items, how many of each label, and which fields every item carries.

In [ ]:
# Count how many items carry each label, one item at a time.
label_counts = {}
for item in rows:
    label = item["label"]
    if label not in label_counts:
        label_counts[label] = 0
    label_counts[label] = label_counts[label] + 1

print("total items:", len(rows))
print("label counts:", label_counts)
print("fields per item:", list(rows[0].keys()))

Now we look at three whole items, to see the shape of one.

Where a track carries a `context` (the whole passage a sentence came from), it is shortened here so it does not bury everything else. That only changes what is **printed** — `rows` itself is untouched.

In [ ]:
for item in rows[:3]:
    preview = dict(item)          # a copy, so trimming it changes nothing
    if preview.get("context"):
        preview["context"] = preview["context"][:70] + " …"
    print(preview)
    print("---")

## Step 5 — Save it

Three short cells: check that `config.yaml` agrees which track this is, check the shape of every item, then write the file.

**First, a safety check.** `POOL_PATH` is built from the `track:` line in `config.yaml`. If that still says another track, saving now would write l2_errors data into a file belonging to something else — and everything downstream would run perfectly on the wrong data. The first sign of trouble would be labels that make no sense in notebook 03, by which point two people have annotated forty items.

If this cell stops you: open `config.yaml`, set `track:` to `l2_errors`, save it, then re-run the SETUP cell at the top of this notebook.

In [ ]:
if TRACK not in ['l2_errors', 'l2_error_detection']:
    raise RuntimeError(
        "config.yaml says  track: " + str(TRACK) + "  but this is the l2_errors "
        "notebook, so saving now would put l2_errors data into "
        + POOL_PATH.name + ", which belongs to another track.\n"
        "Open config.yaml, set  track: to one of l2_errors · l2_error_detection,"
        " save it, then re-run the SETUP cell at the top of this notebook.")

print("config.yaml agrees: this is the", TRACK, "track.")

**Now we check the shape of every item.** Everything downstream — the sampling, the annotation sheet, the scoring — assumes each item has an `id`, a `text` and a `label`. A pool that breaks that assumption does not fail here; it fails in notebook 03, after two people have annotated forty items.

`validate` says nothing when all is well. Silence is the pass.

In [ ]:
validate(rows)
print("All", len(rows), "items have an id, a text and a label.")

**Now we write the pool** into your group's Drive folder, under the exact name notebook 02 will look for. Both notebooks get that name from `config.yaml`, so there is nothing to copy or paste between them.

In [ ]:
import json

POOL_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(POOL_PATH, "w", encoding="utf-8") as f:
    json.dump(rows, f, ensure_ascii=False, indent=2)
print("Saved", len(rows), "items to", POOL_PATH)

# For the binary version, set  track: l2_error_detection  in config.yaml before running this, so POOL_PATH becomes l2_error_detection_pool.json and the two versions do not overwrite each other. Notebook 04 then reads prompts/l2_error_detection.txt, which is already there as a baseline.

## What you just built, and what happens to it

This is the **pool** — everything usable in the corpus, with its natural label imbalance intact. It is **not** your gold set, and its labels are **not** your labels: they are the original corpus authors' judgment, and you have not yet agreed with them about anything.

What those labels are for is narrow, and worth being precise about:

1. **Stratifying the draw** in notebook 02 — you cannot sample evenly across classes without knowing what the classes are.
2. **A comparison** in notebook 03 — once you have annotated blind and adjudicated, `compare_to_published` shows you every item where your group landed somewhere different. That gap is evidence, and one of the more interesting things you can put in a report.

They are never the answer key you score the model against. That file does not exist yet — you make it in notebook 03.

---

**Next:** open `02_sample.ipynb`. It reads `POOL_PATH` — the file the cell above just wrote, in your group's Drive folder. Nothing to copy, nothing to paste: that path is the handoff, and both notebooks get it from the same `config.yaml`.